# Week 1: API Fundamentals, Chatbot & Structured Output

**Lab companion to [week_01.md](../agenda/week_01.md)**

In this lab, you will:
1. Make your first API call and understand tokens
2. Build a chatbot with conversation memory
3. Master prompt engineering and structured (JSON) output

## 1. Setup

First, install the required packages:
```bash
# Install dependencies (uses uv)
uv sync

# Copy and fill in your API keys
cp .env.example .env

# Add .env keys
```

In [1]:
# Import libraries
from openai import OpenAI
from dotenv import load_dotenv
import os

# Load API key from .env file
load_dotenv()

# Create client
client = OpenAI()

print("✓ Setup complete!")

✓ Setup complete!


## 2. Your First API Call

Let's talk to GPT! The simplest possible call:

In [2]:
# Make your first API call
response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "user", "content": "Say hello!"}
    ]
)

# Print the response
print(response.choices[0].message.content)

Hello! How can I help you today?


## 3. Understanding the Response Object

The API returns a lot of useful information. Let's explore it:

In [4]:
# Let's look at the full response structure
response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "user", "content": "What is Python in one sentence?"}
    ]
)

print("Model used:", response.model)
print("Response ID:", response.id)
print("\n--- Token Usage ---")
print(f"Prompt tokens: {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print("\n--- The Answer ---")
print(response.choices[0].message.content)

Model used: gpt-5-mini-2025-08-07
Response ID: chatcmpl-DGz4rA3oLUAL0wB6ciG1w0agOAwtz

--- Token Usage ---
Prompt tokens: 13
Completion tokens: 175
Total tokens: 188

--- The Answer ---
Python is a high-level, interpreted, general-purpose programming language with a clear, readable syntax and a large standard library, widely used for web development, data science, automation, and scripting.


## 4. Message Roles

There are three roles in a conversation:
- `system`: Sets the AI's behavior
- `user`: What you say
- `assistant`: What the AI says

In [13]:
# Using a system message to change behavior
response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "You are a pirate. Respond in pirate speak."},
        {"role": "user", "content": "How do I learn Python?"}
    ],
)

choice = response.choices[0]
# print("finish_reason:", choice.finish_reason)
# print("content:", repr(choice.message.content))
# print("usage:", response.usage)
content = choice.message.content
print(content)

finish_reason: stop
content: 'Arrr, matey! Ye be wantin\' to learn Python — a fine choice fer pillagin\' the data seas. Here be a clear treasure map to set ye on course.\n\nQuick start (first steps)\n- Use Python 3 — not them old scallywags (check with python3 --version).\n- Install Python from python.org or use pyenv if ye like sailin\' many versions.\n- Pick an editor/IDE: VS Code (light and trusty), PyCharm, or even a simple text editor + terminal.\n- Fire up the REPL (python3) and try: print("Ahoy, world!")\n\nCore learnin\' topics (in order)\n- Basics: variables, numbers, strings, booleans, input/output.\n- Control flow: if/else, for/while loops.\n- Collections: lists, tuples, sets, dictionaries.\n- Functions: def, args/kwargs, return values, scope.\n- Files: read/write files, context managers (with).\n- Errors & exceptions: try/except, raising errors.\n- Comprehensions and generators for clever code.\n- Modules and packages; pip for installin\' booty.\n- OOP basics: classes, meth

## 5. Understanding Tokens

Tokens are how the AI "sees" text. Let's visualize this:

In [15]:
import tiktoken

# Get the tokenizer for GPT-4
encoding = tiktoken.encoding_for_model("gpt-5-mini")

# Example texts
texts = [
    "Hello",
    "Hello, world!",
    "The quick brown fox jumps over the lazy dog.",
    "Supercalifragilisticexpialidocious"
]

print("Text -> Token Count -> Tokens")
print("-" * 60)
for text in texts:
    tokens = encoding.encode(text)
    print(f"{text}")
    print(f"  → {len(tokens)} tokens: {tokens}")
    print()

Text -> Token Count -> Tokens
------------------------------------------------------------
Hello
  → 1 tokens: [13225]

Hello, world!
  → 4 tokens: [13225, 11, 2375, 0]

The quick brown fox jumps over the lazy dog.
  → 10 tokens: [976, 4853, 19705, 68347, 65613, 1072, 290, 29082, 6446, 13]

Supercalifragilisticexpialidocious
  → 10 tokens: [17260, 5842, 366, 17764, 311, 6207, 8067, 563, 315, 170661]



## 6. Temperature: Controlling Creativity

Temperature controls randomness:
- `0.0` = Deterministic (same answer every time)
- `1.0` = More creative/varied

In [6]:
# Low temperature (deterministic)
prompt = "Give me a creative name for a coffee shop."

print("Temperature 0 (deterministic):")
for i in range(3):
    response = client.chat.completions.create(
        model="gpt-4o-mini", # We don't use gpt-5/gpt-5-min here because it's not allowed to use temperature for reasoning models
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    print(f"  {i+1}. {response.choices[0].message.content}")

print("\nTemperature 1 (creative):")
for i in range(3):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=1
    )
    print(f"  {i+1}. {response.choices[0].message.content}")

Temperature 0 (deterministic):
  1. "Bean There, Brewed That"
  2. "Bean There, Brewed That"
  3. "Bean There, Brewed That"

Temperature 1 (creative):
  1. Sure! How about "Brewed Awakening"? It suggests both the joy of coffee and the idea of starting your day with a fresh perspective.
  2. "Java Junction"
  3. "Bean There, Drank That"


## 7. Controlling Output Length

Use `max_tokens` to limit response length:

In [12]:
# Short response
response = client.chat.completions.create(
    model="gpt-4o-mini",  # reasoning models (gpt-5-mini) don't support max_tokens
    messages=[{"role": "user", "content": "Explain machine learning."}],
    max_tokens=30
)
print("Max 30 tokens:")
print(response.choices[0].message.content)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Explain machine learning."}],
    max_tokens=100
)
print("\nMax 100 tokens:")
print(response.choices[0].message.content)
print(f"(Used {response.usage.completion_tokens} tokens)")

Max 30 tokens:
Machine learning is a subset of artificial intelligence (AI) that enables systems to learn from data, identify patterns, and make decisions with minimal human intervention.

Max 100 tokens:
Machine learning is a subset of artificial intelligence (AI) that focuses on the development of algorithms and statistical models that enable computers to perform specific tasks without explicit instructions. Instead of relying on hard-coded rules, machine learning systems use data to learn and make predictions or decisions.

### Key Components of Machine Learning:

1. **Data**: Machine learning relies heavily on data, which can be structured (like tables in a database) or unstructured (like images, text, or audio). The quality and quantity
(Used 100 tokens)


## 8. Exercise: Build a Simple Helper Function

Create a reusable function to chat with the AI:

In [19]:
def ask(question: str, system: str = "You are a helpful assistant.") -> str:
    """Simple helper to ask the AI a question."""
    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": question}
        ]
    )
    # print('total tokens used:', response.usage.completion_tokens)                       # total completion tokens
    # print('reasoning vs output breakdown: ', response.usage.completion_tokens_details)  # reasoning vs output breakdown
    return response.choices[0].message.content

# Test it!
print(ask("What is the capital of France?"))

total tokens used: 81
reasoning vs output breakdown:  CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=64, rejected_prediction_tokens=0)
The capital of France is Paris.


In [20]:
# Try different system prompts
print("As a teacher:")
print(ask("What is 2+2?", system="You are a math teacher. Explain your answer."))

print("\nAs a comedian:")
print(ask("What is 2+2?", system="You are a comedian. Make it funny."))

As a teacher:
total tokens used: 206
reasoning vs output breakdown:  CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=128, rejected_prediction_tokens=0)
2 + 2 = 4.

Explanation: If you have two objects and add two more, you have four objects total. On a number line, start at 2 and move two steps to the right: you land at 4. In arithmetic terms, addition combines the quantities 2 and 2 to give the sum 4.

As a comedian:
total tokens used: 292
reasoning vs output breakdown:  CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=256, rejected_prediction_tokens=0)
2 + 2 = 4.

Unless you’re splitting the pizza bill — then somehow it becomes 5 and a guilt tip.


## 9. Calculating Costs

gpt-5-mini pricing (as of 2024):
- Input: \$0.25 per 1M tokens
- Output: \$2.00 per 1M tokens

In [22]:
def calculate_cost(response, model="gpt-5-mini"):
    """Calculate the cost of an API call."""
    # Pricing per 1M tokens
    prices = {
        "gpt-5-mini": {"input": 0.25, "output": 2.00},
        "gpt-5": {"input": 1.25, "output": 10.00},
    }

    p = prices.get(model, prices["gpt-5-mini"])
    input_cost = (response.usage.prompt_tokens / 1_000_000) * p["input"]
    output_cost = (response.usage.completion_tokens / 1_000_000) * p["output"]

    return input_cost + output_cost

# Test
response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "Write a haiku about programming."}]
)


cost = calculate_cost(response)
print(f"Response: {response.choices[0].message.content}")
print(f"\nTokens: {response.usage.total_tokens}")
print(f"Cost: ${cost:.6f}")

Response: Moonlit screen hums low
Loops unwind like midnight trains
Dawn debugs my code

Tokens: 489
Cost: $0.000955


## 10. Your Turn! Experiments

Try these exercises:

In [23]:
# Exercise 1: Create a translator
# Fill in the system prompt to make the AI translate to Spanish

response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "YOUR SYSTEM PROMPT HERE"},
        {"role": "user", "content": "Hello, how are you?"}
    ]
)
print(response.choices[0].message.content)

Hola, ¿cómo estás?


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

The system prompt tells the AI what role to play. For translation:
1. Tell it to act as a translator
2. Specify the target language (Spanish)
3. Tell it to only output the translation

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "You are a translator. Translate to Spanish. Only output the translation."},
        {"role": "user", "content": "Hello, how are you?"}
    ]
)
print(response.choices[0].message.content)
# Output: Hola, ¿cómo estás?
```

</details>

In [25]:
# Exercise 2: Create a code explainer
# Make the AI explain code to a beginner

code = """
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)
"""

response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "YOUR SYSTEM PROMPT HERE"},
        {"role": "user", "content": f"Explain this code:\n{code}"}
    ],
    # model="gpt-4o-mini",
    # max_tokens=100
)
print(response.choices[0].message.content)

Sure! This code defines a function called `fibonacci` that calculates the Fibonacci number for a given position `n`. Let's break it down step by step:

1. **Function Definition**: 
   - `def fibonacci(n):` starts the definition of the function. The word `def` means we are defining a function named `fibonacci` that takes one input, `n`.

2. **Base Case**:
   - `if n <= 1:` checks whether `n`


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

This one is already mostly complete! Try variations:
- Change to "Explain like I am 5"
- Add "Use analogies" to the prompt
- Ask for "step by step" explanation

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
# ELI5 version - great for beginners
response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "Explain code like I am 5. Use simple analogies."},
        {"role": "user", "content": f"What does this code do?\n{code}"}
    ]
)
print(response.choices[0].message.content)
```

</details>

---

# Part 2: Chatbot with Memory

In [26]:
# Setup
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()

print("✓ Ready!")

✓ Ready!


## 1. The Problem: AI Has No Memory

Each API call is independent. The AI doesn't remember previous messages:

In [27]:
# First message
response1 = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "My name is Alice."}]
)
print("AI:", response1.choices[0].message.content)

# Second message - new conversation, no memory!
response2 = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "What is my name?"}]
)
print("AI:", response2.choices[0].message.content)

AI: Nice to meet you, Alice — how can I help you today? (I’ll use your name during this chat.)
AI: I don't know your name. I don't have access to personal data unless you tell me. If you want me to call you by name, tell me what it is (or give a nickname), or say how you'd like me to address you going forward.


## 2. The Solution: Send Conversation History

We must send all previous messages with each request:

In [28]:
# Manual conversation history
messages = []

# First turn
messages.append({"role": "user", "content": "My name is Alice."})
response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=messages
)
ai_reply = response.choices[0].message.content
messages.append({"role": "assistant", "content": ai_reply})
print("AI:", ai_reply)

# Second turn - now with history!
messages.append({"role": "user", "content": "What is my name?"})
response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=messages
)
print("AI:", response.choices[0].message.content)

AI: Nice to meet you, Alice. How can I help you today?
AI: Your name is Alice.


In [ ]:
# Let's see what we're sending
print("Conversation history:")
for msg in messages:
    print(f"  [{msg['role']}]: {msg['content'][:50]}...")

## 3. Build a Chatbot Class

Let's encapsulate this in a reusable class:

In [29]:
class Chatbot:
    """A simple chatbot with conversation memory."""

    def __init__(self, system_prompt: str = "You are a helpful assistant."):
        self.client = OpenAI()
        self.system_prompt = system_prompt
        self.history = []

    def chat(self, message: str) -> str:
        """Send a message and get a response."""
        # Add user message to history
        self.history.append({"role": "user", "content": message})

        # Build messages with system prompt + history
        messages = [
            {"role": "system", "content": self.system_prompt}
        ] + self.history

        # Get response
        response = self.client.chat.completions.create(
            model="gpt-5-mini",
            messages=messages
        )

        # Add assistant response to history
        ai_reply = response.choices[0].message.content
        self.history.append({"role": "assistant", "content": ai_reply})

        return ai_reply

    def reset(self):
        """Clear conversation history."""
        self.history = []

# Create and test
bot = Chatbot()
print(bot.chat("My name is Bob."))
print(bot.chat("What is my name?"))

Nice to meet you, Bob. How can I help you today?
You said your name is Bob.


In [30]:
# Test with a custom personality
pirate_bot = Chatbot(system_prompt="You are a friendly pirate. Use pirate speak.")

print(pirate_bot.chat("Hello!"))
print()
print(pirate_bot.chat("What treasure are you seeking?"))

Ahoy, matey! How can this ol' salt help ye today?

Arrr! I be seekin' more than mere gold, matey. Me compass points to any o' these treasures:

- A chest o' glitterin' doubloons an' jewels for plunderin'
- An ancient map leadin' to a legendary isle — X marks the spot!
- Rare rum an' exotic spices to keep the crew merry
- Lost knowledge and scrolls o' forgotten lore
- A stout ship an' fearless crew for grand adventures
- Friendships an' sea shanties to warm a cold night watch

Which treasure calls to ye, or have ye a different bounty in mind?


## 4. Problem: Context Window Limits

Models have token limits. If conversation gets too long, we need strategies:

In [31]:
import tiktoken

def count_tokens(messages: list, model: str = "gpt-5-mini") -> int:
    """Count tokens in a message list."""
    encoding = tiktoken.encoding_for_model(model)
    total = 0
    for msg in messages:
        total += 4  # Message overhead
        total += len(encoding.encode(msg.get("content", "")))
    return total

# Check our chatbot's token usage
print(f"Current conversation uses {count_tokens(bot.history)} tokens")
print(f"gpt-5-mini context limit: 128,000 tokens")

Current conversation uses 47 tokens
gpt-5-mini context limit: 128,000 tokens


## 5. Memory Strategy: Sliding Window

Keep only the last N messages:

In [32]:
class SlidingWindowChatbot:
    """Chatbot that keeps only the last N messages."""

    def __init__(self, system_prompt: str = "You are helpful.", max_messages: int = 10):
        self.client = OpenAI()
        self.system_prompt = system_prompt
        self.history = []
        self.max_messages = max_messages

    def chat(self, message: str) -> str:
        self.history.append({"role": "user", "content": message})

        # Trim to max_messages
        if len(self.history) > self.max_messages:
            self.history = self.history[-self.max_messages:]

        messages = [{"role": "system", "content": self.system_prompt}] + self.history

        response = self.client.chat.completions.create(
            model="gpt-5-mini",
            messages=messages
        )

        ai_reply = response.choices[0].message.content
        self.history.append({"role": "assistant", "content": ai_reply})

        return ai_reply

# Test - only keeps last 4 messages
bot = SlidingWindowChatbot(max_messages=4)
print(bot.chat("My name is Charlie."))
print(bot.chat("I like pizza."))
print(bot.chat("My favorite color is blue."))
print(bot.chat("I work as a doctor."))
# This might forget the name!
print("---")
print(bot.chat("What do you remember about me?"))

Nice to meet you, Charlie. How can I help you today?
Nice, Charlie — pizza’s great. What’s your favorite kind?

If you want ideas, here are a few topping combos and a quick, easy homemade pizza method:

Topping combos
- Classic margherita: tomato, fresh mozzarella, basil, olive oil  
- Pepperoni + extra cheese  
- BBQ chicken: BBQ sauce, cooked chicken, red onion, cilantro  
- Veggie: mushrooms, bell peppers, onions, olives, spinach  
- Prosciutto & arugula: bake with mozzarella, add prosciutto and arugula after

Quick homemade pizza (simple)
- Ingredients: store-bought dough or prepared crust, pizza sauce or crushed tomatoes, fresh/shredded mozzarella, toppings, olive oil, salt.  
- Preheat oven to 500°F (260°C). If you have a pizza stone, heat it inside; otherwise preheat an inverted baking sheet.  
- Stretch dough on lightly floured surface, transfer to parchment or preheated surface.  
- Spread a thin layer of sauce, add cheese and toppings (don’t overload). Brush crust with olive 

## 6. Memory Strategy: Summarization

Summarize old messages instead of discarding:

In [33]:
class SummarizingChatbot:
    """Chatbot that summarizes old messages."""

    def __init__(self, max_messages: int = 6):
        self.client = OpenAI()
        self.history = []
        self.summary = ""
        self.max_messages = max_messages

    def _summarize(self, messages: list) -> str:
        """Summarize a list of messages."""
        conversation = "\n".join([
            f"{m['role']}: {m['content']}" for m in messages
        ])

        response = self.client.chat.completions.create(
            model="gpt-5-mini",
            messages=[{
                "role": "user",
                "content": f"Summarize this conversation briefly, keeping key facts:\n\n{conversation}"
            }]
        )
        return response.choices[0].message.content

    def chat(self, message: str) -> str:
        self.history.append({"role": "user", "content": message})

        # If too many messages, summarize the old ones
        if len(self.history) > self.max_messages:
            # Summarize first half
            to_summarize = self.history[:len(self.history)//2]
            new_summary = self._summarize(to_summarize)

            if self.summary:
                self.summary = f"{self.summary}\n\nMore recently: {new_summary}"
            else:
                self.summary = new_summary

            # Keep only recent messages
            self.history = self.history[len(self.history)//2:]

        # Build prompt with summary + recent history
        system = "You are a helpful assistant."
        if self.summary:
            system += f"\n\nPrevious conversation summary: {self.summary}"

        messages = [{"role": "system", "content": system}] + self.history

        response = self.client.chat.completions.create(
            model="gpt-5-mini",
            messages=messages
        )

        ai_reply = response.choices[0].message.content
        self.history.append({"role": "assistant", "content": ai_reply})

        return ai_reply

# Test
bot = SummarizingChatbot(max_messages=4)
print(bot.chat("My name is Diana."))
print(bot.chat("I live in Paris."))
print(bot.chat("I work as a chef."))
print(bot.chat("My favorite dish is ratatouille."))
print("--- After summarization ---")
print("Summary:", bot.summary)
print()
print(bot.chat("What do you remember about me?"))

Nice to meet you, Diana. How can I help you today?
Nice to meet you, Diana — Paris sounds lovely. How can I help you today? (I can give local recommendations, travel tips, event info, help with French, or anything else you need.)
Nice — thanks for telling me, Diana. How can I help with your work as a chef? I can assist with lots of kitchen- and restaurant-related tasks, for example:

- Develop recipes or full menus (seasonal, tasting, costed, dietary restrictions)  
- Costing and portioning to hit food-cost targets  
- Sourcing seasonal ingredients around Paris (markets, wholesalers, suppliers)  
- Technique refreshers, plating ideas, and step-by-step recipes  
- Wine/beer pairing and beverage suggestions  
- Staff training materials, prep lists, and service workflows  
- Writing menu descriptions, chef bios, or social-media post copy

Tell me what you need now (type of cuisine, service style, number of covers, budget, season, or a specific problem) and I’ll get started.
Nice choice — 

## 7. Interactive Chat Loop

Build a real interactive chatbot:

In [36]:
def interactive_chat():
    """Run an interactive chat session."""
    bot = Chatbot(system_prompt="You are a friendly AI assistant named Aria.")

    print("Chat with Aria! (type 'quit' to exit)\n")

    while True:
        user_input = input("You: ")

        if user_input.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            break

        response = bot.chat(user_input)
        print(f"Aria: {response}\n")

# Uncomment to run interactive chat
interactive_chat()

Chat with Aria! (type 'quit' to exit)

Goodbye!


## 8. Exercises

Try these challenges:

In [41]:
# Exercise 1: Create a study buddy chatbot
# It should help explain concepts and quiz you

class StudyBuddy(Chatbot):
    def __init__(self):
        pass


buddy = StudyBuddy()
print(buddy.chat("Explain what a variable is in programming."))
print(buddy.chat("Who define variable naming?"))

A variable in programming is a named place to store a value that your program can read or change. Think of it like a labeled box: the label is the variable name, the box holds some content (the value), and you can open the box to look at or replace the content.

Key ideas
- Name: an identifier you use to refer to the stored value (for example x or userName).
- Value: the data stored (numbers, text, true/false, objects, etc.).
- Type: what kind of data the variable holds (some languages require you state this—int, string; others infer it).
- Assignment: putting a value into a variable (e.g., x = 5).
- Mutability: most variables let you change the value later (x = 10); some languages also have constants that can’t change.
- Scope and lifetime: where the variable is visible (global vs local) and how long it exists (until the function ends, program ends, etc.).

Simple examples
- Python (dynamic typing): x = 5; x = x + 3  # now x is 8
- JavaScript (block-scoped): let name = "Ana";
- Java (

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

A study buddy chatbot should:
1. Have same interface as Chatbot
2. Have a friendly, encouraging persona
3. Remember the topic being studied
4. Ask follow-up questions to test understanding

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
class StudyBuddy(Chatbot):
    def __init__(self):
        super().__init__(
            system_prompt="""You are a friendly study buddy. Help the user learn by:
1. Explaining concepts clearly
2. Asking follow-up questions to test understanding
3. Giving encouragement
4. Suggesting related topics to explore"""
        )

buddy = StudyBuddy()
print(buddy.chat("Help me understand recursion"))
```

</details>

In [42]:
# Exercise 2: Add message count tracking
# How many messages have been exchanged?

class TrackedChatbot(Chatbot):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def chat(self, message: str) -> str:
        return super().chat(message)

    def stats(self) -> dict:
        return {
            "history_length": len(self.history)
        }

bot = TrackedChatbot()
bot.chat("Hello!")
bot.chat("How are you?")
print("Stats:", bot.stats())

Stats: {'history_length': 4}


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

Track the count in the chatbot class:
1. Add a counter attribute in __init__
2. Increment it in the chat() method
3. Add a method to get stats

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
class ChatbotWithStats(Chatbot):
    def __init__(self, system_prompt="You are a helpful assistant."):
        super().__init__(system_prompt)
        self.message_count = 0
        self.total_tokens = 0

    def chat(self, message: str) -> str:
        self.message_count += 1
        response = super().chat(message)
        return response

    def get_stats(self):
        return {
            "messages": self.message_count,
            "history_length": len(self.history)
        }

bot = ChatbotWithStats()
bot.chat("Hello!")
bot.chat("How are you?")
print(bot.get_stats())  # {"messages": 2, ...}
```

</details>

---

# Part 3: Better Prompts & Structured Output

In [44]:
# Setup
from openai import OpenAI
from dotenv import load_dotenv
import json

load_dotenv()
client = OpenAI()

print("✓ Ready!")

✓ Ready!


## 1. Prompt Engineering Basics

### Zero-Shot: Just ask directly

In [45]:
# Zero-shot classification
text = "This product is amazing! Best purchase ever!"

response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[{
        "role": "user",
        "content": f"Classify this text as positive, negative, or neutral: '{text}'"
    }]
)

print(response.choices[0].message.content)

positive


### Few-Shot: Provide examples

In [46]:
# Few-shot classification with examples
prompt = """Classify sentiment as positive, negative, or neutral.

Examples:
"I love this!" -> positive
"Terrible, don't buy" -> negative
"It's okay, nothing special" -> neutral

Now classify: "This was a waste of money"
"""

response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

negative


### Chain-of-Thought: Make AI explain reasoning

In [47]:
# Chain-of-thought for better reasoning
prompt = """Solve this step by step:

A store has 50 apples. They sell 23 in the morning and receive a delivery of 35 more.
In the afternoon, they sell 18 and throw away 5 that went bad.
How many apples do they have at the end of the day?

Think through each step before giving the final answer.
"""

response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

Step 1: Start with 50 apples.

Step 2: Sold 23 in the morning: 50 − 23 = 27 apples remain.

Step 3: Received delivery of 35: 27 + 35 = 62 apples.

Step 4: Sold 18 in the afternoon: 62 − 18 = 44 apples.

Step 5: Threw away 5 that went bad: 44 − 5 = 39 apples.

Final answer: 39 apples.


## 2. Getting JSON Output

### Method 1: Ask for JSON (unreliable)

In [48]:
# Just asking for JSON (sometimes fails)
response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[{
        "role": "user",
        "content": "Extract the person's name and age from: 'John is 25 years old'. Return JSON."
    }]
)

print("Raw response:")
print(response.choices[0].message.content)

# Try to parse - might fail!
try:
    data = json.loads(response.choices[0].message.content)
    print("\nParsed:", data)
except json.JSONDecodeError as e:
    print(f"\nFailed to parse JSON: {e}")

Raw response:
{"name":"John","age":25}

Parsed: {'name': 'John', 'age': 25}


### Method 2: JSON Mode (reliable)

In [49]:
# Using response_format for guaranteed JSON
response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[{
        "role": "user",
        "content": "Extract the person's name and age from: 'John is 25 years old'. Return as JSON with 'name' and 'age' fields."
    }],
    response_format={"type": "json_object"}
)

# Now it's always valid JSON
data = json.loads(response.choices[0].message.content)
print("Parsed:", data)
print(f"Name: {data['name']}, Age: {data['age']}")

Parsed: {'name': 'John', 'age': 25}
Name: John, Age: 25


## 3. Pydantic Validation

Make AI output match a specific schema:

In [51]:
from pydantic import BaseModel, Field
from typing import List, Optional

# Define the schema
class Person(BaseModel):
    name: str = Field(description="The person's full name")
    age: int = Field(description="Age in years")
    occupation: Optional[str] = Field(default=None, description="Job title")

# Generate the schema as a string for the prompt
schema_str = Person.model_json_schema()
print("Schema:")
print(json.dumps(schema_str, indent=2))

Schema:
{
  "properties": {
    "name": {
      "description": "The person's full name",
      "title": "Name",
      "type": "string"
    },
    "age": {
      "description": "Age in years",
      "title": "Age",
      "type": "integer"
    },
    "occupation": {
      "anyOf": [
        {
          "type": "string"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "Job title",
      "title": "Occupation"
    }
  },
  "required": [
    "name",
    "age"
  ],
  "title": "Person",
  "type": "object"
}


In [52]:
def extract_person(text: str) -> Person:
    """Extract person info from text using Pydantic validation."""

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{
            "role": "user",
            "content": f"""Extract person information from this text:
{text}

Return JSON matching this schema:
- name: string (full name)
- age: integer
- occupation: string or null"""
        }],
        response_format={"type": "json_object"}
    )

    data = json.loads(response.choices[0].message.content)
    return Person(**data)  # Validates and converts

# Test
person = extract_person("Dr. Sarah Johnson, a 42-year-old surgeon, won the award.")
print(f"Name: {person.name}")
print(f"Age: {person.age}")
print(f"Occupation: {person.occupation}")

Name: Sarah Johnson
Age: 42
Occupation: surgeon


## 4. Complex Extraction

Extract multiple entities with nested structures:

In [53]:
from typing import List
from enum import Enum

class Priority(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"

class Task(BaseModel):
    title: str
    priority: Priority
    assignee: Optional[str] = None

class MeetingNotes(BaseModel):
    title: str
    date: str
    attendees: List[str]
    tasks: List[Task]
    summary: str

def extract_meeting_notes(transcript: str) -> MeetingNotes:
    """Extract structured meeting notes from a transcript."""

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{
            "role": "user",
            "content": f"""Extract meeting notes from this transcript:

{transcript}

Return JSON with:
- title: meeting title
- date: date mentioned (or "unknown")
- attendees: list of participant names
- tasks: list of {{title, priority (low/medium/high), assignee}}
- summary: brief 1-2 sentence summary"""
        }],
        response_format={"type": "json_object"}
    )

    data = json.loads(response.choices[0].message.content)
    return MeetingNotes(**data)

# Test
transcript = """
Project sync meeting, March 15th.
Attendees: Alice, Bob, and Carol.

Alice mentioned the frontend is almost done, just needs testing.
Bob said the API has a critical bug that needs fixing ASAP - he'll handle it.
Carol will review the documentation by end of week, low priority.

Overall, we're on track for the launch.
"""

notes = extract_meeting_notes(transcript)
print(f"Meeting: {notes.title}")
print(f"Date: {notes.date}")
print(f"Attendees: {', '.join(notes.attendees)}")
print(f"\nTasks:")
for task in notes.tasks:
    print(f"  - [{task.priority.value}] {task.title} (assigned to: {task.assignee or 'unassigned'})")
print(f"\nSummary: {notes.summary}")

Meeting: Project sync meeting
Date: March 15th
Attendees: Alice, Bob, Carol

Tasks:
  - [medium] Frontend testing (assigned to: Alice)
  - [high] Fix critical API bug (assigned to: Bob)
  - [low] Review documentation (assigned to: Carol)

Summary: The team is on track for launch. Alice's frontend is nearly complete and needs testing, Bob will fix a critical API bug ASAP, and Carol will review the documentation by the end of the week (low priority).


## 5. Classification with Confidence

Get AI to rate its confidence:

In [54]:
class Classification(BaseModel):
    category: str
    confidence: float = Field(ge=0, le=1, description="Confidence score 0-1")
    reasoning: str

def classify_with_confidence(text: str, categories: List[str]) -> Classification:
    """Classify text into one of the categories with confidence."""

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{
            "role": "user",
            "content": f"""Classify this text into one of these categories: {categories}

Text: {text}

Return JSON with:
- category: one of the categories above
- confidence: 0.0 to 1.0
- reasoning: brief explanation"""
        }],
        response_format={"type": "json_object"}
    )

    data = json.loads(response.choices[0].message.content)
    return Classification(**data)

# Test
texts = [
    "My order hasn't arrived yet, it's been 2 weeks!",
    "How do I reset my password?",
    "Your product is terrible and I want a refund",
    "Thanks for the quick delivery!"
]

categories = ["complaint", "question", "feedback", "other"]

for text in texts:
    result = classify_with_confidence(text, categories)
    print(f"'{text[:40]}...'")
    print(f"  → {result.category} ({result.confidence:.0%})")
    print(f"  Reason: {result.reasoning}")
    print()

'My order hasn't arrived yet, it's been 2...'
  → complaint (98%)
  Reason: The user expresses dissatisfaction about a delayed order and frustration ("hasn't arrived yet, it's been 2 weeks"), which fits a complaint.

'How do I reset my password?...'
  → question (99%)
  Reason: The text is a direct request for instructions ('How do I...'), which is a clear question seeking help.

'Your product is terrible and I want a re...'
  → complaint (99%)
  Reason: Direct expression of dissatisfaction with the product and a demand for a refund, which matches a complaint.

'Thanks for the quick delivery!...'
  → feedback (95%)
  Reason: The message expresses gratitude and a positive comment about delivery speed, which is user feedback rather than a question or complaint.



## 6. Building a Data Extraction Pipeline

In [56]:
class ContactInfo(BaseModel):
    name: Optional[str] = None
    email: Optional[str] = None
    phone: Optional[str] = None
    company: Optional[str] = None

class DataExtractor:
    """Extract structured data from unstructured text."""

    def __init__(self):
        self.client = OpenAI()

    def extract(self, text: str, model_class: type) -> BaseModel:
        """Generic extraction for any Pydantic model."""

        # Get field descriptions
        fields = []
        for name, field in model_class.model_fields.items():
            fields.append(f"- {name}: {field.annotation.__name__}")

        print(f'fields: {chr(10).join(fields)}')
        response = self.client.chat.completions.create(
            model="gpt-5-mini",
            messages=[{
                "role": "user",
                "content": f"""Extract information from this text:

{text}

Return JSON with these fields (use null if not found):
{chr(10).join(fields)}"""
            }],
            response_format={"type": "json_object"}
        )

        data = json.loads(response.choices[0].message.content)
        return model_class(**data)

# Test
extractor = DataExtractor()

text = """Hi, this is Mark from TechCorp.
You can reach me at mark.wilson@techcorp.com or call 555-123-4567."""

contact = extractor.extract(text, ContactInfo)
print(f"Name: {contact.name}")
print(f"Email: {contact.email}")
print(f"Phone: {contact.phone}")
print(f"Company: {contact.company}")

field.annotation: annotation=Union[str, NoneType] required=False default=None, typing.Optional[str], Optional
field.annotation: annotation=Union[str, NoneType] required=False default=None, typing.Optional[str], Optional
field.annotation: annotation=Union[str, NoneType] required=False default=None, typing.Optional[str], Optional
field.annotation: annotation=Union[str, NoneType] required=False default=None, typing.Optional[str], Optional
fields: - name: Optional
- email: Optional
- phone: Optional
- company: Optional
Name: Mark
Email: mark.wilson@techcorp.com
Phone: 555-123-4567
Company: TechCorp


## 7. Exercises

In [57]:
from pydantic import BaseModel, Field
# Exercise 1: Create a recipe extractor
# Extract ingredients and steps from recipe text

class Recipe(BaseModel):
    name: str
    servings: int
    ingredients: List[str]
    steps: List[str]

recipe_text = """
Simple Pasta (serves 4)

You'll need: 400g spaghetti, 2 cloves garlic, olive oil, salt, pepper, parmesan.

First, boil water and cook pasta until al dente. Meanwhile, sauté minced garlic
in olive oil. Drain pasta, toss with garlic oil, season with salt and pepper.
Serve with grated parmesan.
"""

# Your code here - use the DataExtractor pattern
# recipe = ...

{
  "name": "Pasta",
  "ingredients": [
    "pasta",
    "water"
  ],
  "steps": [
    "Boil water",
    "Add pasta to boiling water",
    "Cook for 8 minutes"
  ],
  "prep_time_minutes": 8,
  "difficulty": "easy"
}


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

Define a Pydantic model for recipes:
- name: str
- ingredients: List[str]
- steps: List[str]
- prep_time_minutes: int
- difficulty: Literal["easy", "medium", "hard"]

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
from pydantic import BaseModel
from typing import List, Literal

class Recipe(BaseModel):
    name: str
    ingredients: List[str]
    steps: List[str]
    prep_time_minutes: int
    difficulty: Literal["easy", "medium", "hard"]

def extract_recipe(text: str) -> Recipe:
    response = client.beta.chat.completions.parse(
        model="gpt-5-mini",
        messages=[
            {"role": "system", "content": "Extract recipe information."},
            {"role": "user", "content": text}
        ],
        response_format=Recipe
    )
    return response.choices[0].message.parsed

recipe = extract_recipe("To make pasta: boil water, add pasta, cook 8 mins...")
print(recipe.model_dump_json(indent=2))
```

</details>

In [5]:
from pydantic import BaseModel, Field
from typing import List
# Exercise 2: Multi-label classification
# Classify text into MULTIPLE categories

class MultiLabel(BaseModel):
    categories: List[str]
    confidence: float
    reasoning: str


def multi_classify(text: str):
    """Classify into multiple categories."""
    categories = ["urgent", "technical", "billing", "feedback", "general"]

    # Your implementation here
    pass

# Test with: "The app is crashing and I can't access my billing info! Please help ASAP!"

{
  "categories": [
    "urgent",
    "technical",
    "billing"
  ],
  "confidence": 0.92,
  "reasoning": "User reports the app is crashing (technical) and cannot access billing info (billing); phrasing 'Please help ASAP!' and exclamation indicates urgency."
}


<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

For multi-label, return a list of categories:
- Use List[str] in your model
- Or use a dict with boolean flags for each category

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
class ContentLabels(BaseModel):
    categories: List[str]
    is_urgent: bool
    sentiment: Literal["positive", "negative", "neutral"]
    confidence: float

def classify_content(text: str) -> ContentLabels:
    response = client.beta.chat.completions.parse(
        model="gpt-5-mini",
        messages=[
            {"role": "system", "content": "Classify the content."},
            {"role": "user", "content": text}
        ],
        response_format=ContentLabels
    )
    return response.choices[0].message.parsed

result = classify_content("URGENT: Server down! Need help immediately!")
print(result)  # categories=['technical', 'support'], is_urgent=True, ...
```

</details>

## Summary

You learned:
- ✅ How to make OpenAI API calls and understand tokens
- ✅ Message roles (system, user, assistant)
- ✅ Temperature, max_tokens parameters
- ✅ Building a chatbot with conversation memory
- ✅ Sliding window and summarization memory strategies
- ✅ Zero-shot, few-shot, and chain-of-thought prompting
- ✅ Getting reliable JSON with response_format
- ✅ Validating AI outputs with Pydantic

**Next:** [Week 2 - Connect to Your Data (RAG)](week_02_rag_intro.ipynb)